# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("{}: {}".format(metadata.name, metadata.description))

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

**Note:** All entities are referenced by their `@id` fields.

In [ ]:
# Retrieve record sets
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"Record Set @id: {rs['@id']}, name: {rs.get('name', '-')}")
    fields = rs.get('field', [])
    if fields:
        print("  Fields:")
        for f in fields:
            if isinstance(f, dict):
                print(f"   - {f.get('@id', f)}: {f.get('name', '-')}")
            else:
                print(f"   - {f}")
    columns = rs.get('column', [])
    if columns:
        print("  Columns:")
        for c in columns:
            if isinstance(c, dict):
                print(f"   - {c.get('@id', c)}: {c.get('name', '-')}")
            else:
                print(f"   - {c}")
    print()

# For demonstration, we'll select the first record set
if record_sets:
    main_record_set_id = record_sets[0]['@id']
else:
    main_record_set_id = None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.
Use the record set and field `@id`s from the overview above.

In [ ]:
# Collect data from all available record sets
dataframes = {}
record_set_ids = []
# Ensure record_set_ids are extracted properly
for rs in record_sets:
    rs_id = rs['@id']
    record_set_ids.append(rs_id)
    try:
        records_iter = dataset.records(record_set=rs_id)
        records = list(records_iter)
        if records:
            dataframes[rs_id] = pd.DataFrame(records)
        else:
            dataframes[rs_id] = pd.DataFrame()
    except Exception as e:
        print(f"Error loading records from {rs_id}: {e}")

# Preview the columns for the main record set
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Columns in record set {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    print("No main record set available for preview.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section includes:
- Filtering records where GAD-7 total score (`@id`: `https://api.app.sen.science/frontiers/7160411/gad7_total`) exceeds a threshold
- Normalizing that score
- Grouping by level of education (`@id`: `https://api.app.sen.science/frontiers/7160411/level_of_education`)

**Make sure you reference fields by their `@id`!**

In [ ]:
# Specify field IDs
gad7_total_id = 'https://api.app.sen.science/frontiers/7160411/gad7_total'
level_of_education_id = 'https://api.app.sen.science/frontiers/7160411/level_of_education'

record_set_id = main_record_set_id
df = dataframes[record_set_id]

# Filter records with GAD-7 > threshold
threshold = 10
if gad7_total_id in df.columns:
    filtered_df = df[df[gad7_total_id] > threshold].copy()
    print(f"Filtered records with {gad7_total_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize the GAD-7 total score
    filtered_df[f"{gad7_total_id}_normalized"] = (
        filtered_df[gad7_total_id] - filtered_df[gad7_total_id].mean()
    ) / filtered_df[gad7_total_id].std()

    print(f"Normalized {gad7_total_id} for filtered records:")
    display(filtered_df[[gad7_total_id, f"{gad7_total_id}_normalized"]].head())

    # Group by level of education
    if level_of_education_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(level_of_education_id)[gad7_total_id].mean().reset_index()
        print(f"Average GAD-7 score grouped by {level_of_education_id}:")
        display(grouped_df.head())
    else:
        print(f"Field {level_of_education_id} not found for grouping.")
else:
    print(f"Field {gad7_total_id} not found in dataframe columns.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 total score
if gad7_total_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[gad7_total_id].dropna(), bins=20, kde=True)
    plt.title("Distribution of GAD-7 Total Score")
    plt.xlabel('GAD-7 Total Score')
    plt.ylabel('Count')
    plt.show()

# Boxplot of GAD-7 by level of education
if gad7_total_id in df.columns and level_of_education_id in df.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(x=df[level_of_education_id], y=df[gad7_total_id])
    plt.title('GAD-7 Score by Level of Education')
    plt.xlabel('Level of Education')
    plt.ylabel('GAD-7 Total Score')
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Mental Health Survey Dataset from Kilifi County provides detailed information on psychological symptoms, demographics, and assessment scores (GAD-7, PHQ-9, PSQ).
- Using `mlcroissant`, we efficiently loaded and inspected dataset metadata and contents, referencing all entities by their `@id`.
- Initial exploratory analysis reveals patterns in GAD-7 scores relative to education; further analysis may consider additional fields and relationships.
- This notebook demonstrates transparent, reproducible AI-ready data exploration practices.